# LoRA Fine-Tuning with Unsloth: A Complete Tutorial

This notebook demonstrates how to fine-tune a Large Language Model (LLM) using **LoRA** (Low-Rank Adaptation) with the Unsloth library on NVIDIA GPUs.

---

## What is LoRA?

**LoRA** (Low-Rank Adaptation) is a parameter-efficient fine-tuning technique that significantly reduces the number of trainable parameters while maintaining model quality.

### How LoRA Works:

Instead of updating all model weights, LoRA:
1. **Freezes** the original pre-trained weights
2. **Injects** small trainable matrices (adapters) into specific layers
3. **Trains only** these adapters (typically <1% of total parameters)

```
Original: W₀ (frozen)
LoRA:     W₀ + BA (B and A are trainable low-rank matrices)
```

### Benefits of LoRA:
- **~10x less memory** than full fine-tuning
- **Faster training** due to fewer parameters to optimize
- **Easy model switching**: swap adapters without reloading the base model
- **Minimal quality loss** compared to full fine-tuning

---

## Why Use Unsloth?

**Unsloth** provides optimized implementations that offer:
- **2x faster training** compared to standard implementations
- **Up to 70% less memory usage**
- **Zero accuracy degradation**
- Full compatibility with HuggingFace ecosystem

---

## Requirements

- NVIDIA GPU (GTX 1070+, RTX series, A100, H100)
- CUDA installed
- Python 3.8+

---

## 1. Environment Setup

First, we need to install Unsloth and its dependencies. This includes:

| Package | Purpose |
|---------|--------|
| **bitsandbytes** | Quantization support |
| **accelerate** | HuggingFace's distributed training library |
| **xformers** | Memory-efficient attention implementations |
| **peft** | Parameter-Efficient Fine-Tuning library |
| **trl** | Transformer Reinforcement Learning, includes SFTTrainer |
| **unsloth** | The optimization library we're using |

In [ ]:
# Install Unsloth and required dependencies
# Note: --no-deps prevents dependency conflicts
# Uncomment if you have already installed the dependencies or if you are running this on Colab.

# %pip install --no-deps bitsandbytes accelerate xformers==0.0.29.post3 peft trl triton cut_cross_entropy unsloth_zoo matplotlib
# %pip install sentencepiece protobuf "datasets>=3.4.1" huggingface_hub hf_transfer
# %pip install --no-deps unsloth

### 1.1 Verify CUDA Availability

Before proceeding, let's verify that CUDA is available and check our GPU specifications. LoRA with Unsloth requires a CUDA-enabled NVIDIA GPU for optimal performance.

In [ ]:
import torch
import logging

logging.basicConfig(level=logging.INFO)

# Check CUDA availability
logging.info(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    logging.info(f"GPU: {torch.cuda.get_device_name(0)}")
    logging.info(
        f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
    logging.info(f"CUDA Version: {torch.version.cuda}")
else:
    logging.warning("CUDA is not available. Training will be slower on CPU.")

---

## 2. Model Configuration

We'll use a **pre-quantized 4-bit model** from Unsloth. These models are already optimized for memory efficiency.

### Model Selection:

Unsloth provides pre-quantized models for popular architectures:

| Model | Size | Use Case |
|-------|------|----------|
| `unsloth/Llama-3.2-1B-Instruct-bnb-4bit` | 1B | Fast prototyping |
| `unsloth/Llama-3.2-3B-Instruct-bnb-4bit` | 3B | Good balance |
| `unsloth/Llama-3.1-8B-Instruct-bnb-4bit` | 8B | Higher quality |

### Configuration Parameters:

- **max_seq_length**: Maximum input sequence length. Longer = more context but more memory.
- **load_in_4bit**: Enable 4-bit quantization to reduce memory (recommended: True)
- **dtype**: Computation precision. `None` means auto-detect optimal dtype.

In [ ]:
from unsloth import FastLanguageModel

# ============================================
# Model Configuration
# ============================================

# Pre-quantized 4-bit model from Unsloth
# The '-bnb-4bit' suffix indicates bitsandbytes 4-bit quantization
MODEL_NAME = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit"

# Maximum sequence length for training
# Supports RoPE Scaling for extensions beyond training length
MAX_SEQ_LENGTH = 2048

# Enable 4-bit quantization for memory efficiency
LOAD_IN_4BIT = True

# Auto-detect optimal dtype based on GPU:
# - Float16 for older GPUs (T4, V100)
# - Bfloat16 for newer GPUs (Ampere+: A100, RTX 3090+)
DTYPE = None

logging.info("Configuration:")
logging.info(f"  Model: {MODEL_NAME}")
logging.info(f"  Max Sequence Length: {MAX_SEQ_LENGTH}")
logging.info(f"  4-bit Quantization: {LOAD_IN_4BIT}")

### 2.1 Load the Model with Unsloth

Unsloth's `FastLanguageModel.from_pretrained()` automatically applies optimizations:
- Fast weight loading
- Memory-efficient attention
- Optimized kernels for inference and training

In [ ]:
# Load Model with Unsloth optimizations
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=DTYPE,
    load_in_4bit=LOAD_IN_4BIT,
)

logging.info("\n✓ Model loaded successfully!")

---

## 3. LoRA Adapter Configuration

Now we add **LoRA adapters** to the model. These small, trainable matrices are the key to parameter-efficient fine-tuning.

### Understanding LoRA Parameters:

| Parameter | Description | Recommended | Impact |
|-----------|-------------|-------------|--------|
| `r` (rank) | Dimension of LoRA matrices | 8-64 | Higher = more capacity, more memory |
| `lora_alpha` | Scaling factor | Equal to `r` | Controls adapter influence |
| `lora_dropout` | Regularization dropout | 0-0.1 | Prevents overfitting |
| `target_modules` | Which layers to adapt | All QKV + MLP | More = more flexibility |

### Target Modules Explained:

In Transformer models, we typically target:

**Attention Layers:**
- `q_proj`: Query projection - what to look for
- `k_proj`: Key projection - what information is available  
- `v_proj`: Value projection - the actual information
- `o_proj`: Output projection - combines attention outputs

**MLP Layers:**
- `gate_proj`: Controls information flow
- `up_proj`: Projects to higher dimension
- `down_proj`: Projects back to model dimension

In [ ]:
# ============================================
# LoRA Parameters
# ============================================

# Rank: Higher = more trainable parameters, more expressive
# Common values: 8, 16, 32, 64, 128
LORA_R = 16

# Alpha: Scaling factor for LoRA updates
# Common practice: set equal to rank
LORA_ALPHA = 16

# Dropout: Regularization (0 is optimized by Unsloth)
LORA_DROPOUT = 0

# Bias: Type of bias to train ("none" is optimized)
LORA_BIAS = "none"

# Target all attention and MLP projections for maximum adaptability
TARGET_MODULES = [
    "q_proj",     # Query projection
    "k_proj",     # Key projection
    "v_proj",     # Value projection
    "o_proj",     # Output projection
    "gate_proj",  # Gate projection (MLP)
    "up_proj",    # Up projection (MLP)
    "down_proj",  # Down projection (MLP)
]

logging.info("LoRA Configuration:")
logging.info(f"  Rank (r): {LORA_R}")
logging.info(f"  Alpha: {LORA_ALPHA}")
logging.info(f"  Target modules: {len(TARGET_MODULES)} layers")

### 3.1 Apply LoRA Adapters

We use `FastLanguageModel.get_peft_model()` to inject LoRA adapters into the model.

**Important**: The `use_gradient_checkpointing="unsloth"` option enables:
- 30% less VRAM usage
- Support for 2x larger batch sizes
- Optimized backward pass

In [ ]:
# Apply LoRA adapters to the model
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=TARGET_MODULES,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias=LORA_BIAS,
    # Unsloth's optimized gradient checkpointing:
    # - Uses 30% less VRAM
    # - Fits 2x larger batch sizes
    use_gradient_checkpointing="unsloth",
    random_state=3407,  # For reproducibility
    use_rslora=False,   # Rank-Stabilized LoRA (optional advanced feature)
    loftq_config=None,  # LoftQ initialization (optional)
)

logging.info("\n✓ LoRA adapters applied!")

### 3.2 Analyze Trainable Parameters

One of LoRA's key advantages is that we only train a tiny fraction of parameters. Let's see the numbers.

In [ ]:
# Calculate and display parameter counts
trainable_params = sum(p.numel()
                       for p in model.parameters() if p.requires_grad)
all_params = sum(p.numel() for p in model.parameters())
trainable_percent = 100 * trainable_params / all_params

logging.info("=" * 50)
logging.info("PARAMETER SUMMARY")
logging.info("=" * 50)
logging.info(f"Trainable parameters: {trainable_params:,}")
logging.info(f"Total parameters:     {all_params:,}")
logging.info(f"Trainable:            {trainable_percent:.4f}%")
logging.info("=" * 50)

---

## 4. Dataset Preparation

For this tutorial, we use the **FineTome-100k** dataset, a high-quality instruction-following dataset in ShareGPT format.

### ShareGPT Format:

ShareGPT format structures conversations as a list of message turns:

```json
{
  "conversations": [
    {"role": "user", "content": "Hello!"},
    {"role": "assistant", "content": "Hi! How can I help?"}
  ]
}
```

This format is widely used and compatible with most training pipelines.

In [ ]:
from datasets import load_dataset

# Load the FineTome-100k dataset
dataset = load_dataset("mlabonne/FineTome-100k", split="train")

logging.info(f"✓ Dataset loaded!")
logging.info(f"Number of examples: {len(dataset):,}")

---

## 5. Chat Template and Prompt Formatting

Each model family has its own **chat template** - a specific format for structuring conversations. Using the correct template is crucial for good performance.

### Llama 3.1/3.2 Template Structure:

```
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

{system_message}<|eot_id|><|start_header_id|>user<|end_header_id|>

{user_message}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

{assistant_response}<|eot_id|>
```

In [ ]:
from typing import Any, cast
from unsloth.chat_templates import get_chat_template, standardize_sharegpt

# Apply the Llama 3.1 chat template to the tokenizer
tokenizer = get_chat_template(
    tokenizer,
    chat_template="llama-3.1",
)

# Standardize dataset to ShareGPT format
dataset = standardize_sharegpt(dataset)

logging.info("✓ Chat template applied: llama-3.1")

### 5.1 Create Formatting Function

This function converts each conversation into the model's expected format.

In [ ]:
def formatting_prompts_func(examples):
    """
    Convert conversations to the model's expected text format.

    This function:
    1. Takes a batch of conversations
    2. Applies the chat template to each
    3. Returns formatted text strings
    """
    convos = examples["conversations"]
    texts = [
        tokenizer.apply_chat_template(
            convo,
            tokenize=False,
            add_generation_prompt=False
        )
        for convo in convos
    ]
    return {"text": texts}


# Apply formatting to the entire dataset
dataset = dataset.map(formatting_prompts_func, batched=True)

logging.info("✓ Dataset formatted successfully!")

### 5.2 Inspect Formatted Example

Let's look at a formatted example to understand the transformation.

In [ ]:
# View an example before and after formatting
item = cast(Any, dataset[5])

logging.info("ORIGINAL CONVERSATION:")
logging.info("-" * 40)
logging.info(item["conversations"])

logging.info("\nFORMATTED TEXT:")
logging.info("-" * 40)
logging.info(item["text"])

---

## 6. Training Configuration

Now we configure the training parameters optimized for LoRA fine-tuning.

### Key Parameters:

| Parameter | Description | Typical Values |
|-----------|-------------|----------------|
| `per_device_train_batch_size` | Samples per GPU per step | 1-8 |
| `gradient_accumulation_steps` | Steps before weight update | 4-16 |
| `learning_rate` | How fast the model learns | 1e-5 to 5e-4 |
| `warmup_steps` | Gradual LR increase | 10-100 |
| `optim` | Optimizer choice | `adamw_8bit` for memory |

### Memory vs Speed Trade-offs:

- **Higher batch size** → More memory, faster training
- **More accumulation steps** → Same effective batch, less memory
- **8-bit optimizer** → Less memory, same quality

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments, DataCollatorForSeq2Seq
from unsloth import is_bfloat16_supported

# ============================================
# Training Configuration
# ============================================

OUTPUT_DIR = "./outputs"           # Checkpoint directory
NUM_TRAIN_EPOCHS = 3               # Number of training epochs
PER_DEVICE_TRAIN_BATCH_SIZE = 2    # Batch size per GPU
GRADIENT_ACCUMULATION_STEPS = 4    # Accumulate gradients
# Effective batch size = 2 * 4 = 8

LEARNING_RATE = 2e-4               # Learning rate for LoRA
WARMUP_STEPS = 10                  # LR warmup steps
MAX_STEPS = 60                     # Max steps (for demo; use -1 for full)
LOGGING_STEPS = 10                 # Log frequency
OPTIMIZER = "adamw_8bit"           # 8-bit optimizer saves memory

# Auto-detect precision support
use_bf16 = is_bfloat16_supported()
use_fp16 = not use_bf16

logging.info(f"Precision: {'bfloat16' if use_bf16 else 'float16'}")
logging.info(
    f"Effective batch size: {PER_DEVICE_TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS}")

### 6.1 Create Training Arguments

In [ ]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    warmup_steps=WARMUP_STEPS,
    # num_train_epochs=NUM_TRAIN_EPOCHS,  # Uncomment for full training
    max_steps=MAX_STEPS,  # For demo purposes
    learning_rate=LEARNING_RATE,
    fp16=use_fp16,
    bf16=use_bf16,
    logging_steps=LOGGING_STEPS,
    optim=OPTIMIZER,
    weight_decay=0.01,
    lr_scheduler_type="linear",
    seed=3407,
    report_to="none",  # Change to "wandb" for tracking
)

logging.info("✓ Training arguments configured!")

### 6.2 Initialize SFTTrainer

The `SFTTrainer` (Supervised Fine-Tuning Trainer) from TRL is designed for instruction-tuning language models.

In [ ]:
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer),
    tokenizer=tokenizer,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=2,
    packing=False,  # True = faster for short sequences
    args=training_args,
)

logging.info("✓ SFTTrainer initialized!")

---

## 7. Train Only on Assistant Responses

An important optimization: we only compute loss on **assistant responses**, not on user messages. This:
- Focuses learning on generation quality
- Reduces noise in the training signal
- Makes training more efficient

In [ ]:
from unsloth.chat_templates import train_on_responses_only

# Configure training to compute loss only on assistant responses
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|start_header_id|>user<|end_header_id|>\n\n",
    response_part="<|start_header_id|>assistant<|end_header_id|>\n\n",
)

# Begin training
logging.info("Starting LoRA fine-tuning...")
trainer_stats = trainer.train()

logging.info("\n" + "=" * 50)
logging.info("✓ Training completed!")
logging.info(
    f"Training time: {trainer_stats.metrics['train_runtime']:.2f} seconds")

In [ ]:
import matplotlib.pyplot as plt

# Extract loss history
history = trainer.state.log_history

# Filter for loss values
loss_values = [x['loss'] for x in history if 'loss' in x]
steps = [x['step'] for x in history if 'loss' in x]

if len(loss_values) > 0:
    plt.figure(figsize=(10, 6))
    plt.plot(steps, loss_values, marker='o', linestyle='-',
             color='b', label='Training Loss')
    plt.title('LoRA Training Loss Evolution')
    plt.xlabel('Training Steps')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()
else:
    print("No loss history found.")

### 7.1 Visualize Training Loss

Let's visualize the training progress. A consistent decrease in loss indicates successful learning.

---

## 8. Inference: Test Your Fine-Tuned Model

Let's test the model by generating responses. We use `FastLanguageModel.for_inference()` for 2x faster generation.

In [ ]:
from unsloth.chat_templates import get_chat_template

# Prepare for inference
tokenizer = get_chat_template(tokenizer, chat_template="llama-3.1")
FastLanguageModel.for_inference(model)  # Enable 2x faster inference

# Create a test message
messages = [
    {"role": "user", "content": "Continue the Fibonacci sequence: 1, 1, 2, 3, 5, 8,"},
]

# Prepare input
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
).to("cuda")

# Generate response
outputs = model.generate(
    input_ids=inputs,
    max_new_tokens=64,
    use_cache=True,
    temperature=1.5,
    min_p=0.1
)

# Decode and display
text = tokenizer.decode(outputs[0], skip_special_tokens=True)
logging.info(text)

---

## 9. Save the Fine-Tuned Model

### Save Options:

1. **LoRA Adapters Only** (~100MB): Requires base model for inference
2. **Merged Model**: Full model with adapters merged
3. **GGUF Format**: For llama.cpp, Ollama, etc.

### 9.1 Save LoRA Adapters

In [ ]:
model_name = "Llama32_fine_tuned"
model.save_pretrained(model_name)
tokenizer.save_pretrained(model_name)

logging.info(f"✓ Model saved to: {model_name}/")

### 9.2 Export to GGUF Format (Optional)

Uncomment to export for use with llama.cpp or Ollama.

In [ ]:
# Uncomment to export to GGUF
# model.push_to_hub_gguf(model_name, tokenizer, quantization_method="q4_k_m")

---

## 10. Load and Use the Fine-Tuned Model

Here's how to load your fine-tuned model for future inference sessions.

In [ ]:
from transformers import TextStreamer
from unsloth import FastLanguageModel

# Load the fine-tuned model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Llama32_fine_tuned",
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=DTYPE,
    load_in_4bit=LOAD_IN_4BIT,
)
FastLanguageModel.for_inference(model)

# Test with streaming
messages = [
    {"role": "user", "content": "Describe a tall tower in the capital of France."},
]
inputs = tokenizer.apply_chat_template(
    messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
).to("cuda")

text_streamer = TextStreamer(tokenizer, skip_prompt=True)
_ = model.generate(
    input_ids=inputs,
    streamer=text_streamer,
    max_new_tokens=128,
    use_cache=True,
    temperature=1.5,
    min_p=0.1
)

In [ ]:
import time

# Prepare a longer prompt for benchmarking
messages = [
    {"role": "user", "content": "Explain the theory of relativity in simple terms."},
]
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
).to("cuda")

# Warmup run
print("Warming up GPU...")
_ = model.generate(input_ids=inputs, max_new_tokens=20)

# Benchmark run
print("Running benchmark...")
torch.cuda.synchronize()
start_time = time.time()

outputs = model.generate(
    input_ids=inputs,
    max_new_tokens=512,
    use_cache=True,
    temperature=0.7,
    min_p=0.1,
)

torch.cuda.synchronize()
end_time = time.time()

# Calculate metrics
generated_tokens = outputs.shape[1] - inputs.shape[1]
duration = end_time - start_time
tokens_per_sec = generated_tokens / duration

print(f"\nBENCHMARK RESULTS:")
print(f"Total time: {duration:.2f} seconds")
print(f"Tokens generated: {generated_tokens}")
print(f"Speed: {tokens_per_sec:.2f} tokens/sec")

### 10.1 Inference Speed Benchmark

Let's benchmark the model's generation speed to ensure it meets performance requirements.

---

## 11. GPU Memory Analysis

Let's analyze our GPU memory usage to see the efficiency of LoRA fine-tuning.

In [ ]:
if torch.cuda.is_available():
    logging.info("\n" + "=" * 50)
    logging.info("GPU MEMORY STATISTICS")
    logging.info("=" * 50)

    allocated = torch.cuda.memory_allocated(0) / 1024**3
    reserved = torch.cuda.memory_reserved(0) / 1024**3
    max_allocated = torch.cuda.max_memory_allocated(0) / 1024**3
    total = torch.cuda.get_device_properties(0).total_memory / 1024**3

    logging.info(f"\nMemory Usage:")
    logging.info(f"  • Current:   {allocated:.2f} GB")
    logging.info(f"  • Reserved:  {reserved:.2f} GB")
    logging.info(f"  • Peak:      {max_allocated:.2f} GB")
    logging.info(f"  • Total GPU: {total:.2f} GB")
    logging.info(f"  • Used:      {(max_allocated/total)*100:.1f}%")

    torch.cuda.empty_cache()
    logging.info("\n✓ GPU cache cleared!")

---

## 🎉 Congratulations!

You have successfully completed LoRA fine-tuning with Unsloth!

### Key Takeaways:

1. **LoRA freezes base weights** and only trains small adapter matrices
2. **<1% of parameters trained** but maintains quality
3. **Unsloth provides 2x speedup** and 70% memory reduction
4. **Train on responses only** for more efficient learning

### Next Steps:

- Try **QLoRA** (Quantized LoRA) for even lower memory usage
- Experiment with different `lora_r` values
- Fine-tune on your own custom dataset
- Export to GGUF for local deployment

### Resources:

- [Unsloth GitHub](https://github.com/unslothai/unsloth)
- [LoRA Paper](https://arxiv.org/abs/2106.09685)
- [HuggingFace PEFT](https://huggingface.co/docs/peft)